# 06: Volt-VAr-induced curtailment: Methods A, B and C

Three methods that **bracket** the same quantity:

| | | |
|---|---|---|
| **A** | apparent-limit symptom scan | **upper bound**. Assumes every symptom interval was sun-limited |
| **B** | counterfactual attribution | **lower bound**. Counts only what clear-sky GHI confirms |
| **C** | derating-flag corroboration | not an estimate. Independent *label* to test A and B against |

A and B are reported as a **range** and are not reconciled. (note that averaging them wouldinvent a number neither method supports).

## sign caveat:
the reactive sign is not fully resolved (213 sites fit as-delivered, 106 fit flipped).

`exclude_polarity_suspect=True` (the default) drops the sites `se_adverse` flags as
`polarity_suspect` before scanning. That is a **cohort restriction**, not a correction —
it removes sites whose direction cannot be trusted rather than silently flipping them.

In [7]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_current, *_current.parents)
                  if (p / "solar_edge").is_dir() and (p / "bms_sa_review").is_dir()), None)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from solar_edge.config import se_config as C
from solar_edge.lib import se_store, se_contract as contract, se_params
pd.set_option("display.max_columns", None); pd.set_option("display.width", 220)
con = se_store.connect()
config, params = se_params.CONFIG, se_params.PARAMS
from solar_edge.lib import se_curtailment as cu
from solar_edge.lib import se_adverse as adv

adverse = adv.classify_adverse_sites(con, config)
display(adverse.adverse_class.value_counts())
display(contract.manifest(config, params).query("section in ('detection','basis')"))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


adverse_class
adverse_but_inactive    613
not_adverse             554
genuinely_adverse       314
polarity_suspect        109
Name: count, dtype: int64

,section,setting,value
27,basis,rating_basis,s_99
28,basis,empirical_limit_basis,s_99
29,basis,tolerance_basis,s_99
30,basis,tolerance_fraction,0.04
31,basis,voltage_aggregation,mean
32,basis,reactive_orientation,as_delivered
33,basis,site_nonconf_threshold,0.1
34,detection,voltage band,240 - 253 V
35,detection,peak hours (AEST),11:00 - 14:00
36,detection,require_apparent_limit_symptom,True


## Method A: apparent-limit symptom scan

An interval is flagged when the inverter is **absorbing** reactive power **and** sitting on its apparent-power circle:

```
Q_kvar < 0  AND  sqrt(P² + Q²) >= s_limit − tol × capacity
```

The proxy energy is the **headroom displacement**, `s_limit − sqrt(s_limit² − Q²)`: The kW of circle room that reactive absorption consumed.

**Two biases, both upward.** 

- It assumes the inverter would have used that headroom, which is only true when the sun was available. 
- And `s_limit` is `s_99`, an *observed* p99: a site that never approached its true inverter limit gets a low `s_limit`, so the test fires more readily. 

Method A here is an even looser upper bound givan it has no nameplate to anchor to.

In [8]:
method_a = cu.method_a_site_year(con, config, params, adverse=adverse)
display(cu.method_a_summary(method_a, config).T)

context = cu.eligible_context(con, config, params, adverse=adverse)
print(f"Eligible: {int(context.n_eligible_intervals.sum()):,} intervals "
      f"across {context.site_alias.nunique():,} sites")
print(f"Measured energy in those intervals: {context.measured_kWh.sum():,.0f} kWh")

,0,1
cohort,single-phase,three-phase
eligible_sites,1058,413
symptom_sites,190,410
eligible_intervals,11312789,4242130
absorbing_intervals,1444825,4173727
symptom_intervals,338769,605300
symptom_pct_of_eligible,2.9946,14.2688
headroom_displacement_kWh,4556.7,2483.2
symptom_with_derating_flag,266815,359401


Eligible: 15,554,919 intervals across 1,471 sites
Measured energy in those intervals: 5,494,423 kWh


##  Method B: counterfactual attribution

**Requires the GHI counterfactual.** The cell below raises a clear error if `se_uncurtailedpv` is absent rather than returning zeros.

Four evidence tiers, each strictly narrower than the last:

```
1  absorbing Q                                Q < 0
2  + apparent-limit symptom                   on the S-circle
3  + counterfactual above measured-Q headroom uncurtailed_P > pmax
4  + attributable displacement > 0            the reportable number
```

In [9]:
try:
    method_b = cu.method_b_site_year(con, config, params, adverse=adverse)
    display(cu.method_b_summary(method_b, config).T)
    display(cu.evidence_tiers(method_b))
except FileNotFoundError as exc:
    method_b = None
    print("Method B not runnable yet:\n"); print(exc)

,0,1
cohort,single-phase,three-phase
eligible_sites,1058,413
counterfactual_covered_sites,1058,413
eligible_intervals,11312789,4242130
counterfactual_covered_intervals,7462335,2801287
counterfactual_coverage_pct,65.96,66.03
tier1_absorbing,1444825,4173727
tier2_symptom,338769,605300
tier3_cf_above_headroom,96739,218174
tier4_attributed,58592,75390


,evidence_tier,n_intervals,n_sites,pct_of_tier1
0,Tier 1: absorbing Q,5618552,629,100.00
1,Tier 2: + apparent-limit symptom,944069,600,16.80
2,Tier 3: + counterfactual above measured-Q head...,314913,594,5.60
3,Tier 4: + attributable displacement,133982,472,2.38


## Method C: derating-flag corroboration

SolarEdge reports `derating_active` per interval. 

This is **not a third estimate**, it is an independent label for the thing Method A is trying to infer.

**Read precision, not recall.** The raw flag is `1.0` or NULL, never `0.0`, so "not
derating" and "not reported" are indistinguishable. *Of the intervals the inverter says
it was derating, what fraction did Method A catch?* is sound. *Of the intervals Method A
missed, how many were really derating?* is unanswerable.

The flag is also not Volt-VAr specific — thermal limits, DC clipping and export control
all set it. Agreement corroborates "something limited output", not the mechanism.

In [10]:
confusion = cu.method_c_confusion(con, config, params, adverse=adverse)
display(confusion)
counts = confusion.attrs["counts"]
print(f"Method A symptom AND derating flag : {counts['tp']:,}")
print(f"Method A symptom, no flag          : {counts['fp']:,}")
print(f"Flag but no Method A symptom       : {counts['fn']:,}")
print(f"\nPrecision (interpretable)          : {confusion.attrs['precision']:.4f}")
print("Recall is NOT interpretable — see above.")

,method_a_symptom,derating_active,n_intervals,pct_of_eligible
0,True,True,626216,4.026
1,True,False,317853,2.043
2,False,True,3460823,22.249
3,False,False,11150027,71.682


Method A symptom AND derating flag : 626,216
Method A symptom, no flag          : 317,853
Flag but no Method A symptom       : 3,460,823

Precision (interpretable)          : 0.6633
Recall is NOT interpretable — see above.


### Is the flag tracking Volt-VAr or Volt-Watt?

The discriminator. If the derating rate only climbs above **253 V** it is tracking
Volt-Watt and says nothing about the 240–253 V band where Method A operates. A rise
*within* 240–253 V is what would make it corroborate a Volt-VAr claim.

In [11]:
by_v = cu.method_c_by_voltage(con, config)
display(by_v)

band = by_v[(by_v.v_bin >= 240) & (by_v.v_bin < 253)]
print(f"Derating rate at 240 V: {band.pct_derating.iloc[0]:.2f}%")
print(f"Derating rate at 252 V: {band.pct_derating.iloc[-1]:.2f}%")
print("A rise within this band supports a Volt-VAr reading; a flat line does not.")

,v_bin,n_intervals,n_derating,pct_derating,mean_P_kW,median_Q_kvar
0,235.0,1539649,69255,4.498,1.579,0.1308
1,236.0,2206238,106546,4.829,1.705,0.1285
2,237.0,3034274,158960,5.239,1.852,0.1258
3,238.0,3945880,225884,5.725,2.005,0.1240
4,239.0,4916144,319382,6.497,2.141,0.1239
5,240.0,5962500,440750,7.392,2.268,0.1248
6,241.0,6847161,579907,8.469,2.404,0.1262
7,242.0,7358534,720923,9.797,2.550,0.1286
8,243.0,7286345,828526,11.371,2.741,0.1325
9,244.0,6665134,881063,13.219,2.991,0.1365


Derating rate at 240 V: 7.39%
Derating rate at 252 V: 52.61%
A rise within this band supports a Volt-VAr reading; a flat line does not.


## A vs B vs C

The range, stated as a range.

In [12]:
display(cu.method_comparison(method_a, method_b, confusion, config))

method_a.to_csv(C.ARTEFACT_DIR / "method_a_by_site.csv", index=False)
print(f"-> {C.ARTEFACT_DIR / 'method_a_by_site.csv'}")

,method,bound,sites_flagged,intervals_flagged,energy_kWh,note
0,A -- apparent-limit symptom scan,upper,600.0,944069,7039.9,assumes every symptom interval was sun-limited
1,B -- counterfactual attribution,lower,472.0,133982,3411.5,counterfactual-confirmed only; contaminated tr...
2,C -- derating-flag corroboration,"label, not an estimate",NaN,4087039,NaN,precision of Method A against the flag: 0.663


-> C:\Users\z3553082\OneDrive - UNSW\Documents\GitHub\CICCADA\solar_edge\artefacts\method_a_by_site.csv


## What this establishes

- **Method A** gives an upper-bound curtailment estimate, with both of its upward biases
  stated (sun-limited assumption, and `s_99` as an observed limit).
- **Method C** gives a precision figure for Method A against SolarEdge's own derating
  flag — a validation no Solar Analytics dataset could support.
- **Method B** is coded and will run the moment D12 lands.

Sensitivity for all of this is in `07_sensitivity.ipynb`.